# Etapa 1: Ingestión y Construcción de Datasets Oficiales (SIPSA + EVA)

**Proyecto**: Plataforma Analítica del Mercado de la Papa en Colombia (SIPSA & EVA 2019–2025)  
**Marco Metodológico**: CRISP-DM (Etapa 1: Business / Data Understanding & Ingestion) & Marco PDCO (Fase PLAN / DEVELOPMENT)  
**Estándares**: Clean Architecture, SWEBOK, DAMA-BOK, ISO/IEC 25010  

---

### Objetivos de la Etapa:
1. **Ingestión Mayorista SIPSA**: Procesar los boletines mensuales DANE SIPSA consolidando precios mayoristas mensuales sin nulos.
2. **Ingestión Agrícola Oficial EVA**: Procesar la base agrícola MinAgricultura/UPRA (2019–2025) aislando Papa Criolla y Papa Todas las Variedades.
3. **Estandarización y Persistencia**: Normalizar DIVIPOLA y persistir en la capa `data/CLEANED/` en formato Apache Parquet.


In [9]:
# ==============================================================================
# CELDA 1 OBLIGATORIA: INSTALACIÓN IDEMPOTENTE DE DEPENDENCIAS EN EL KERNEL
# ==============================================================================
import sys

%pip install --quiet --upgrade pip
%pip install --quiet \
    numpy>=1.24.0 \
    pandas>=2.0.0 \
    scipy>=1.10.0 \
    statsmodels>=0.14.0 \
    scikit-learn>=1.3.0 \
    matplotlib>=3.7.0 \
    seaborn>=0.12.0 \
    openpyxl>=3.1.0 \
    pyreadstat>=1.2.0 \
    fastparquet>=2023.8.0 \
    pyarrow>=12.0.0 \
    tabulate>=0.9.0

print(f"[OK] Dependencias verificadas con éxito en Python: {sys.version.split()[0]}")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
[OK] Dependencias verificadas con éxito en Python: 3.13.5


In [10]:
# ==============================================================================
# CONFIGURACIÓN DEL ENTORNO Y CARGA DE MÓDULOS SRC (CLEAN ARCHITECTURE)
# ==============================================================================
from pathlib import Path
import sys

# Incorporar la raíz del proyecto al PYTHONPATH
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np

from src.infrastructure.config import RAW_SIPSA_DIR, CLEANED_DIR
from src.application.ingestion_service import IngestionService
from src.domain.invariants import validar_precio_escalar

print(f"Directorio Raw SIPSA: {RAW_SIPSA_DIR}")
print(f"Directorio de Salida Cleaned: {CLEANED_DIR}")

Directorio Raw SIPSA: C:\Users\ADAN\OneDrive\Documentos\analisispapamercadoorlando\data\RAW\sipsa
Directorio de Salida Cleaned: C:\Users\ADAN\OneDrive\Documentos\analisispapamercadoorlando\data\CLEANED


In [11]:
# ==============================================================================
# EJECUCIÓN DEL SERVICIO DE INGESTIÓN POLÍGLOTA Y UNIFICACIÓN MENSUAL
# ==============================================================================
service = IngestionService(raw_dir=RAW_SIPSA_DIR, output_dir=CLEANED_DIR)
ingestion_service = service  # Alias explícito de compatibilidad
df_mensual, auditoria = service.construir_dataset_unico_nacional()

print("\n--- RESUMEN DE AUDITORÍA DE INGESTIÓN ---")
print(f"Archivos procesados: {auditoria['archivos_procesados']}")
print(f"Total filas mensuales generadas: {auditoria['total_filas_mensuales']:,}")


Iniciando ingestión de 16 periodos crudos de SIPSA...
[1/16] Procesando: 2019_SemI / 2019_SemI.csv
[2/16] Procesando: 2019_SemII / 2019_SemII.csv
[3/16] Procesando: 2020_SemI / 2020_SemI.csv
[4/16] Procesando: 2020_SemII / 2020_SemII.csv
[5/16] Procesando: 2021 ( I Semestre) / 2021 ( I Semestre).csv
[6/16] Procesando: 2021 (II.semestre) / 2021(II Semestre).csv
[7/16] Procesando: 2022 ( I Semestre ) / SIPSA_A_Isem2022.csv
[8/16] Procesando: SIPSA_A_Isem2023 / SIPSA_A_Isem2023.csv
[9/16] Procesando: 2023 ( II Semestre ) / SIPSA_A_IIsem2023.csv
[10/16] Procesando: 2024 (III Cuatrimestre) 4 / SIPSA_A IIIcuatrim2024.csv
[11/16] Procesando: 2025 (I Cuatrimestre) / 2025 (I cuatrimestre).csv
[12/16] Procesando: 2025 (II cuatrimestre) / 2025 (II cuatrimestre).csv
[13/16] Procesando: 2025 (III cuatrimestre) / 2025 (III cuatrimestre).csv
[14/16] Procesando: SIPSA_A_Icuatrim2024 / SIPSA_A Icuatrim2024.csv
[15/16] Procesando: SIPSA_A_IIcuatrim2024 / SIPSA_A IIcuatrim2024.csv
[16/16] Procesando: SIP

In [12]:
# ==============================================================================
# AUDITORÍA DE CALIDAD DAMA-BOK: VERIFICACIÓN DE CERO PRECIOS NULOS
# ==============================================================================
total_filas = len(df_mensual)
nulos_precio = df_mensual["precio_prom_kg"].isna().sum()
menores_cero = (df_mensual["precio_prom_kg"] <= 0).sum()

print("=== CONTROL DE CALIDAD DE PRECIOS ===")
print(f"Total registros en dataset: {total_filas:,}")
print(f"Precios nulos (NaN / None): {nulos_precio} (Tasa: {nulos_precio/total_filas:.4%})")
print(f"Precios <= 0: {menores_cero}")

assert nulos_precio == 0, "ERROR CRÍTICO: Se encontraron precios nulos en el dataset consolidado."
assert menores_cero == 0, "ERROR CRÍTICO: Se encontraron precios menores o iguales a cero."
print("\n[PASÓ] ¡INVARIANTE DE CERO PRECIOS NULOS CUMPLIDA AL 100%!")

=== CONTROL DE CALIDAD DE PRECIOS ===
Total registros en dataset: 86,054
Precios nulos (NaN / None): 0 (Tasa: 0.0000%)
Precios <= 0: 0

[PASÓ] ¡INVARIANTE DE CERO PRECIOS NULOS CUMPLIDA AL 100%!


In [13]:
# ==============================================================================
# PERFILAMIENTO DEL EJE TEMPORAL MENSUAL CONTINUO
# ==============================================================================
print("=== COBERTURA TEMPORAL DEL DATASET ÚNICO ===")
resumen_anual = df_mensual.groupby("año").agg(
    meses_observados=("mes", "nunique"),
    registros_mensuales=("precio_prom_kg", "count"),
    volumen_total_ton=("volumen_ingreso_ton", "sum"),
    precio_promedio_general=("precio_prom_kg", "mean")
).reset_index()

display(resumen_anual)

print("\nPrimeros 5 registros del Dataset Único:")
display(df_mensual.head(5))

=== COBERTURA TEMPORAL DEL DATASET ÚNICO ===


,año,meses_observados,registros_mensuales,volumen_total_ton,precio_promedio_general
0,2019,12,12313,1.274046e+06,2340.217263
1,2020,12,12509,1.361764e+06,1926.850588
2,2021,12,11771,1.315735e+06,2958.082843
3,2022,12,11466,1.413381e+06,4322.306370
4,2023,12,12636,1.520792e+06,3817.477522
5,2024,12,12499,1.534588e+06,3031.287378
6,2025,12,12860,1.632362e+06,2807.085149



Primeros 5 registros del Dataset Único:


,fecha_mes,año,mes,divipola_depto,nombre_depto,divipola_mpio,nombre_mpio,mercado_mayorista,alimento,variedad_papa,volumen_ingreso_ton,precio_prom_kg,precio_min_kg,precio_max_kg,num_transacciones
0,2019-01-01,2019,1,05,ANTIOQUIA,05001,MEDELLÍN,"CALI, CAVASA",PAPA,PAPA NEVADA,8.00,2187.94,2012.90,2406.73,1
1,2019-01-01,2019,1,05,ANTIOQUIA,05001,MEDELLÍN,"MEDELLÍN, CENTRAL MAYORISTA DE ANTIOQUIA",PAPA,PAPA CRIOLLA,4.35,3263.66,3001.99,3590.18,12
2,2019-01-01,2019,1,05,ANTIOQUIA,05001,MEDELLÍN,"MONTERÍA, MERCADO DEL SUR",PAPA,PAPA CAPIRA,105.55,2547.91,2338.02,2804.16,37
3,2019-01-01,2019,1,05,ANTIOQUIA,05001,MEDELLÍN,"MONTERÍA, MERCADO DEL SUR",PAPA,PAPA CRIOLLA,690.00,3220.37,2940.44,3570.77,12
4,2019-01-01,2019,1,05,ANTIOQUIA,05002,ABEJORRAL,"MEDELLÍN, CENTRAL MAYORISTA DE ANTIOQUIA",PAPA,PAPA CRIOLLA,1.80,3262.25,3001.27,3588.48,1


In [16]:
# ==============================================================================
# INGESTIÓN DEL DATASET AGROPECUARIO OFICIAL EVA (MINAGRICULTURA / UPRA 2019-2025)
# ==============================================================================
from src.application.ingestion_service import IngestionService
from src.infrastructure.config import RAW_SIPSA_DIR, CLEANED_DIR

ingestion_service = IngestionService(raw_dir=RAW_SIPSA_DIR, output_dir=CLEANED_DIR)
df_eva, auditoria_eva = ingestion_service.procesar_eva()

print("\n=== AUDITORÍA DE INGESTIÓN DATASET EVA ===")
print(f"Total registros municipales-semestrales: {len(df_eva):,}")
print(f"Variedades canónicas aisladas: {df_eva['desagregacion_cultivo'].unique().tolist()}")
print(f"Volumen total de producción: {df_eva['produccion_ton'].sum():,.2f} Toneladas")
print(f"Área sembrada total: {df_eva['area_sembrada_ha'].sum():,.2f} Ha")
print(f"Área cosechada total: {df_eva['area_cosechada_ha'].sum():,.2f} Ha")
print(f"Rendimiento agronómico promedio: {df_eva['rendimiento_ton_ha'].mean():.2f} Ton/Ha")

display(df_eva.head(5))




Iniciando procesamiento de Base Agrícola EVA desde: C:\Users\ADAN\OneDrive\Documentos\analisispapamercadoorlando\data\RAW\eva\20260526_BaseAgricola20192025.xlsx

¡DATASET CANÓNICO EVA AGRÍCOLA GENERADO CON ÉXITO!
Ruta: C:\Users\ADAN\OneDrive\Documentos\analisispapamercadoorlando\data\CLEANED\dataset_eva_agricola_nacional.parquet
Registros Totales Papa en Campo: 5,574
PAPA TODAS LAS VARIEDADES: 3,869
PAPA CRIOLLA: 1,705
Producción Histórica Total: 27,845,454.2 Ton
Área Cosechada Total: 1,279,242.0 Ha
Rendimiento Promedio Nacional: 16.52 t/ha

=== AUDITORÍA DE INGESTIÓN DATASET EVA ===
Total registros municipales-semestrales: 5,574
Variedades canónicas aisladas: ['PAPA TODAS LAS VARIEDADES', 'PAPA CRIOLLA']
Volumen total de producción: 27,845,454.17 Toneladas
Área sembrada total: 1,341,486.44 Ha
Área cosechada total: 1,279,241.99 Ha
Rendimiento agronómico promedio: 16.52 Ton/Ha


,cod_depto,departamento,cod_mpio,municipio,desagregacion_cultivo,cultivo,ciclo_cultivo,grupo_cultivo,subgrupo,año,periodo,area_sembrada_ha,area_cosechada_ha,produccion_ton,rendimiento_ton_ha,nombre_cientifico,cod_cultivo,estado_fisico
0,05,ANTIOQUIA,05001,MEDELLÍN,PAPA TODAS LAS VARIEDADES,Papa,Transitorio,Raíces y tubérculos,Raíces y tubérculos,2019,2019A,40.0,40.0,640.0,16.0,Solanum tuberosum,1080701,En fresco
1,05,ANTIOQUIA,05002,ABEJORRAL,PAPA TODAS LAS VARIEDADES,Papa,Transitorio,Raíces y tubérculos,Raíces y tubérculos,2019,2019A,95.0,90.0,900.0,10.0,Solanum tuberosum,1080701,En fresco
2,05,ANTIOQUIA,05004,ABRIAQUÍ,PAPA TODAS LAS VARIEDADES,Papa,Transitorio,Raíces y tubérculos,Raíces y tubérculos,2019,2019A,0.5,0.5,2.5,5.0,Solanum tuberosum,1080701,En fresco
3,05,ANTIOQUIA,05079,BARBOSA,PAPA TODAS LAS VARIEDADES,Papa,Transitorio,Raíces y tubérculos,Raíces y tubérculos,2019,2019A,30.0,27.0,216.0,8.0,Solanum tuberosum,1080701,En fresco
4,05,ANTIOQUIA,05086,BELMIRA,PAPA CRIOLLA,Papa,Transitorio,Raíces y tubérculos,Raíces y tubérculos,2019,2019A,150.0,120.0,1020.0,8.5,Solanum phureja,1080702,En fresco


### Conclusiones de la Etapa 1:
- **Dataset SIPSA**: El Dataset Único Nacional SIPSA se ha generado y persistido en `data/CLEANED/dataset_sipsa_mensual_nacional.parquet` con cero precios nulos.
- **Dataset EVA**: Se incorporó la base agrícola oficial MinAgricultura/UPRA 2019–2025 en `data/CLEANED/dataset_eva_agricola_nacional.parquet`, aislando *Papa Criolla* y *Papa Todas las Variedades* con rigurosa consistencia agronómica y códigos DIVIPOLA estandarizados.
- Ambos datasets conforman los pilares de oferta mayorista y oferta agrícola en campo para las etapas analíticas posteriores.
